# Read this first 
## Run this Code once to get the data for you local use we are only working on 2024 then we will expand into the other sections

In [ ]:
import fastf1
import pandas as pd
import os
import sqlite3
from tqdm import tqdm

os.makedirs("f1_cache", exist_ok=True)
fastf1.Cache.enable_cache("f1_cache")

In [ ]:
from tqdm import tqdm

all_laps = []

schedule = fastf1.get_event_schedule(2024)

for _, event in tqdm(schedule.iterrows(), desc="2024"):
    try:
        session = fastf1.get_session(2024, event["EventName"], "R")
        session.load(telemetry=False)
        laps = session.laps[["Driver", "LapNumber", "Compound", "TyreLife", "LapTime", "Stint", "LapStartTime"]].copy()
        laps["Year"] = 2024
        laps["Race"] = event["EventName"]
        all_laps.append(laps)
    except Exception as e:
        print(f"Skipped {event['EventName']}: {e}")

all_laps_df = pd.concat(all_laps, ignore_index=True)
all_laps_df = all_laps_df.dropna(subset=["LapTime"])

print(all_laps_df.shape)
all_laps_df

In [ ]:
conn = sqlite3.connect("f1_data.db")
all_laps_df.to_sql("laps_2024", conn, if_exists="replace", index=False)
conn.close()